<a href="https://colab.research.google.com/github/Peeyusj/week19_huggingface/blob/main/week19_huggingface.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from transformers import pipeline


In [ ]:
generator = pipeline('text-generation', model='gpt2')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [ ]:
classifier = pipeline('sentiment-analysis')
output = classifier('I absolutely loved this movie!')
print(output)

In [ ]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained('gpt2')

In [ ]:
text = "Hello, I am learning Hugging Face!"
tokens = tokenizer(text)
print(tokens)

In [ ]:
for id in tokens['input_ids']:
    print(id, '->', tokenizer.decode([id]))

In [ ]:
from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained('gpt2')

In [ ]:
import torch

input_ids = tokenizer("The meaning of life is", return_tensors='pt')
print(input_ids)

In [ ]:
with torch.no_grad():
    outputs = model(**input_ids)

print(type(outputs))
print(outputs.logits.shape)

In [ ]:
next_token_logits = outputs.logits[0, -1, :]
print(next_token_logits.shape)

In [ ]:
next_token_id = next_token_logits.argmax()
print(next_token_id)

In [ ]:
print(tokenizer.decode([407]))

In [ ]:
import matplotlib.pyplot as plt

# get top 10 predicted next tokens
top10 = next_token_logits.topk(10)

labels = [tokenizer.decode([i]) for i in top10.indices]
values = top10.values.tolist()

plt.figure(figsize=(10, 5))
plt.bar(labels, values)
plt.title("Top 10 predicted next tokens after 'The meaning of life is'")
plt.xlabel("Token")
plt.ylabel("Logit score")
plt.ylim(-98, -93)
plt.show()


In [ ]:
for label, value in zip(labels, values):
    print(f"{label}: {value:.2f}")

In [ ]:
import torch.nn.functional as F

probs = F.softmax(next_token_logits, dim=-1)
top10_probs = probs.topk(10)

labels = [tokenizer.decode([i]) for i in top10_probs.indices]
values = top10_probs.values.tolist()

for label, value in zip(labels, values):
    print(f"{label}: {value:.4f}")

In [ ]:
sum(values)

In [ ]:
input_ids = tokenizer("The meaning of life is", return_tensors='pt')

output = model.generate(
    input_ids['input_ids'],
    max_new_tokens=20
)

print(tokenizer.decode(output[0]))

In [ ]:
print(output)
print(output.shape)

In [ ]:
print(tokenizer.decode([198]))

In [ ]:
for id in [1918, 13, 198, 198]:
    print(id, '->', repr(tokenizer.decode([id])))

In [ ]:
# --- Week 19: Complete Manual Inference Pipeline ---

# Step 1: imports - all imports at top of file
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

# Step 2: load tokenizer - downloads vocab.json and merges.txt from HF Hub (or cache)
tokenizer = AutoTokenizer.from_pretrained('gpt2')

# Step 3: load model - downloads model.safetensors (548MB weights) and config.json
model = AutoModelForCausalLM.from_pretrained('gpt2')

# Step 4: tokenize input - converts text to token IDs, return_tensors='pt' gives PyTorch tensors
input_ids = tokenizer("The meaning of life is", return_tensors='pt')

# Step 5: forward pass - torch.no_grad() skips gradient tracking since we're not training
with torch.no_grad():
    outputs = model(**input_ids)  # ** unpacks dict into keyword arguments

# Step 6: get next token logits - [0]=first sentence, [-1]=last position, [:]=all 50257 scores
next_token_logits = outputs.logits[0, -1, :]

# decode single next token - argmax finds index of highest score
next_token_id = next_token_logits.argmax()
print(tokenizer.decode([next_token_id]))

# Step 7: full generation - autoregressive loop, stops at max_new_tokens or end token
output = model.generate(input_ids['input_ids'], max_new_tokens=20)
print(tokenizer.decode(output[0]))  # output[0] strips batch dimension

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


{'input_ids': tensor([[ 464, 3616,  286, 1204,  318]]), 'attention_mask': tensor([[1, 1, 1, 1, 1]])}
<class 'transformers.modeling_outputs.CausalLMOutputWithCrossAttentions'>
torch.Size([1, 5, 50257])
torch.Size([50257])
